# Tissue extractor tutorial

Tutorial for running the tissue extractor.

# should rename this - this isn't the whole tutorial.

Make this file the file for processing (norm - call once, then convert cell to `raw`; reslice - for all images) for slope, intercept images.

## to do for tutorial

- make the base loop a helper function that you can call for each function

- run each function in its own cell using the helper function

- Above is for the multiple subject case

- Also just show how to do it on one subject

In [1]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
import ants

import SUITPy as suit
import SUITPy.atlas as atlas

import nitools as nt
import tissue_extractor as te

from pathlib import Path
import os

# BEFORE RUNNING: UPLOAD TO GITHUB REPO (THIS AND TISSUE EXTRACTOR, UPDATED); THEN RUN

In [2]:
"""
dummy function to test loop; to be replaced with the actual function
"""
def dummy_fcn(subj_id, week):
    print(subj_id, week)

In [2]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [4]:
tissue = 'wm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

In [6]:
te.normalize?

Signature: te.normalize(t1_path, mask_path, results_path, space='SUIT')
Docstring:
June 9: updated with updated suit function
with correct input arg names
and option to choose template space to normalize to
File:      ~/Documents/GitHub/smarts_cerebellum/image_processing/tissue_extractor.py
Type:      function

In [7]:
subj_id = 'CU_2310'
week = 'W0'

results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/' # folder specifies space

t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'


In [8]:
# normalization

# this is the base loop - make helper function. Use this loop to run each function of the tissue_extractor in its own cell.
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue

    if not Path(tissue_path).is_file():
        print(f'{tissue} path does not exist for {subj_id} in week {week}')
        continue
    

    # make a new folder inside subject's week folder for results
    results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/'
    results_path.mkdir(parents=True, exist_ok = True) # exist_ok = True

    mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'

    te.normalize(t1_path, mask_path, results_path, space = 'MNI152NLin2009cSymC')

    print(f'{subj_id} {week} normalization done')

"""
    results = te.normalize(t1_path = t1_path,
                           mask_path = mask_path,
                           results_path = results_path
                           )

    print(f'{subj_id} {week} normalization done')
"""
    
# run this on one subject, see mif it works, stop the loop, upload to repo, and then run all the subjects. Then can upload to repo again.

Normalizing CU_2310_W0_T1 to tpl-MNI152NLin2009cSymC_T1w.nii.gz
Saving the normalized image into CU_2310_W0_T1_space-MNI152NLin2009cSymC.nii.gz
Saving deformation field into CU_2310_W0_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving inverse deformation field into CU_2310_W0_T1_from-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving the Jacobian determinant to CU_2310_W0_T1_to-MNI152NLin2009cSymC_mode-image_detJ.nii.gz
Saving the log-Jacobian determinant to CU_2310_W0_T1_to-MNI152NLin2009cSymC_mode-image_log_detJ.nii.gz
CU_2310 W0 normalization done
Normalizing CU_2310_W4_T1 to tpl-MNI152NLin2009cSymC_T1w.nii.gz
Saving the normalized image into CU_2310_W4_T1_space-MNI152NLin2009cSymC.nii.gz
Saving deformation field into CU_2310_W4_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving inverse deformation field into CU_2310_W4_T1_from-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving the Jacobian determinant to CU_2310_W4_T1_to-MNI152NLin2009cSymC_mode-image_detJ.nii.gz
Saving the lo

"\n    results = te.normalize(t1_path = t1_path,\n                           mask_path = mask_path,\n                           results_path = results_path\n                           )\n\n    print(f'{subj_id} {week} normalization done')\n"

# ask joern
before running this!
In the regression module, should I put it back to voxel coordinates before writing image (intercept and slope)?

Also should talk to Joern about the voxel reg on just the t1 anatomical.

# check the code for the new week_path before running

# restart kernel for updated av function before calling this

In [ ]:
from image_analysis import avg_vol as av

In [ ]:
# CALL FOR WM SEGMENTATION IMAGE IN NATIVE SPACE________change image suffix for other types

betas = [] # store matrices for all subjects

for subj in p_df['subj_id'].unique():
    #betas.append(avg_vol(subj, ref_img))
    refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
    ref_img = f'{anat_dir}/{subj}/{refT1}/c2{subj}_{refT1}_T1.nii'

    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,
                         results_path = f'{anat_dir}/{subj}/'),
                         image_suffix = "wm_native"
                         )
    
    # we don't need this weeks loop, the function does it. Really just need to loop through the subject and get their reference img
    # even better if the function can get the reference image itself.
    
    #week = p_df.loc[(p_df['subj_id']==subj), 'week'].iloc[1]
   



# note
In the present avg_vol function, week paths are for c2; this is terrible, need to fix (i.e. have a tissue_dict, where if tissue = None, then it'll just put nothing in front, so like):

tissue_dict = {
    'gm': c1,
    'wm': c2,
    'csf': c3,
    None: '' # i don't know if this notation would work
}

Otherwise, we can just have an if tissue!=None statement, so if tissue is supplied, it'll use the dict above (minus None coding, which is strange and might not work) to find the correct prefix of the file.

I did the latter.

Should check that this works well on one subject before continuing wiht the rest. ANd need to ask Joern about the voxel coordiantes thing first, but also check the code from nilearn.